In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd
#fileName=os.listdir(path)
#full_path = os.path.join(path, fileName[0])
#csv_path = os.path.join(full_path)
csv_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df_clean = df.copy()
df_clean = df_clean.drop(columns="Order_ID", axis=1)

In [ ]:
# Task 2: Write your code here:
# Analyze missing values
missing_percentage = (df_clean.isnull().sum() / len(df_clean)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

# Delete the missing values
missData = missing_data['Column'].copy().tolist()
print(f'The missing values columns: \n{missData}\n')
# Drop rows where target (price) or key features are missing - can't predict without them
print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset = missData)
print(f"After dropping missing values: {df_clean.shape}")

In [ ]:
df_clean

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
# 1. Identify all categorical columns in X (features).
def encode_categorical_columns(dataframe):
    categorical_cols = dataframe.select_dtypes(include=["object"]).columns
    print("Categorical Columns:", list(categorical_cols))

onehot_encoders = encode_categorical_columns(df_clean)

# 2. Apply One-Hot Encoding to the categorical feature columns in X.
from sklearn.preprocessing import OneHotEncoder
categorical_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()

X_obj = df_clean[categorical_cols]
categorical_cols.append('Delivery_Time')
X_rem = df_clean.drop(categorical_cols, axis=1)
target = df_clean['Delivery_Time']

print(categorical_cols)
le = OneHotEncoder(sparse_output=False)
X_obj_encoded = pd.DataFrame(le.fit_transform(X_obj), columns=le.get_feature_names_out(X_obj.columns))
X_obj_encoded

In [ ]:
# Task 5:
from sklearn.preprocessing import StandardScaler
numerical_cols = X_rem.select_dtypes(include=["int64", "float64"]).columns  ### DON'T SCALE THE TARGET
scaler = StandardScaler()
X_rem[numerical_cols] = scaler.fit_transform(X_rem[numerical_cols])

In [ ]:
# Task 5: Write your code here:
features = X_obj_encoded
X_rem_cols = X_rem.columns.to_list()

for col in X_rem_cols:
  features[col] = X_rem[col]

features.shape


In [ ]:
features

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
X = features
y = target
X

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as mean_absolute_error

lr_mae = []
models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "Ridge Regression": Ridge(alpha=1.0, max_iter=10000),
  "LASSO Regression": Lasso(alpha=1.0,  max_iter=10000),
}

# Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mae': []}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
n_splits = 5  # K=5 Folds
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)

    # Store results
    all_results[model_name]["mae"].append(mae)
    lr_mae.append(mae)

In [ ]:
import numpy as np
for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MAE:  {np.mean(all_results[model_name]['mae']):.4f}")

In [ ]:
# Calculate the baseline predictions (mean of the target)
baseline_pred = np.full_like(y, y.mean())

# Evaluate the baseline
baseline_mae = mean_absolute_error(y, baseline_pred)

print(f"Baseline MAE (using mean target): {baseline_mae:.4f}")


In [ ]:
# Task 1: Write your code here:
from sklearn.linear_model import Ridge, Lasso
coeffs = {}

coeffs['Lasso'] = models['LASSO Regression'].coef_
coeffs['Ridge'] = models['Ridge Regression'].coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(y_pred, "Delivery_Time")

In [ ]:
! pip install CatBoost

In [ ]:
# Task Bonus: Write your code here:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as mean_absolute_error

lr_mae = []
models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost": CatBoostRegressor(verbose=0)
}

# Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mae': []}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
n_splits = 5  # K=5 Folds
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)

    # Store results
    all_results[model_name]["mae"].append(mae)
    lr_mae.append(mae)